# 02 · Python fundamentals

Explore object identity and shared references, duck typing, string methods and
formatting, safe files, imports, and isolated environments. This notebook is
self-contained: it does not depend on variables or functions from chapter 01.

All examples use the standard library. Files and temporary modules are created in
`TemporaryDirectory` contexts and removed automatically. No network access or keyboard
input is required. Exercise starters are safe during **Run All**; complete them before
adding calls. Worked answers are in `../solutions/02_fundamentals_solutions.ipynb`.

**Source:** adapted and expanded from `2_python_fundamentals.pdf`, pages 1–65.


## 1. Brief review: expressions, decisions, loops, functions

Assignment binds names to typed objects. Arithmetic produces values; comparisons
produce Boolean results. Strings and lists can be indexed and sliced. Empty built-in
collections are falsy. Indentation groups statements under `if`, `for`, and `def`.

The following small program brings these ideas together. Simulated input remains
text until explicitly converted. The function returns its result rather than printing
it, which makes it reusable and easy to check. The prime guard matters for 0 and 1.


In [ ]:
from math import isqrt


def is_prime(number):
    if number < 2:
        return False
    for divisor in range(2, isqrt(number) + 1):
        if number % divisor == 0:
            return False
    return True


raw_limit = "10"  # Simulated input(...) result.
limit = int(raw_limit)
primes = []
for number in range(2, limit):
    if not is_prime(number):
        continue
    primes.append(number)
print("Primes:", primes, "reversed:", primes[::-1])
assert primes == [2, 3, 5, 7]
assert not is_prime(1)


## 2. Everything is an object

Integers, strings, `None`, lists, functions, and types are objects. Each object has a
type, a value or state, and an identity. `isinstance(value, type)` tests whether an
object belongs to that type or a subclass. `type(value) is int` is a narrower exact
type check; for example, `True` is an instance of `int` but its exact type is `bool`.

Use the object model to explain behavior, not to memorize a particular interpreter's
memory layout. Object sizes depend on the Python implementation, build, and value.
`sys.getsizeof` reports a shallow implementation-dependent size; it does not total
all objects reachable from a container.


In [ ]:
import sys

for value in [4, "Hello", None, [1, 2, 3], is_prime, int]:
    print(type(value).__name__, "is an object:", isinstance(value, object))
print(isinstance(True, int), type(True) is int)  # True False
small = [1, 2, 3]
print("Reported shallow list size:", sys.getsizeof(small), "bytes")
# No exact size assertion: this is an observation, not a language guarantee.


## 3. Identity, names, and assignment

`id(object)` is an integer identity token, stable and unique **among objects alive
at the same time**. Python may reuse an identity after an object has been destroyed.
Treat it as an opaque token. Some CPython versions use an address internally; neither
an address interpretation nor particular identity values are portable guarantees.

A name is a label bound to an object. `alias = original` adds a second label; it does
not copy the object. Changing a shared mutable object is visible through both names.
Rebinding one label points that label somewhere else.

```text
original ─┐
          ├──> one list object: ["red", "blue"]
alias ────┘
```


In [ ]:
original = ["red", "blue"]
alias = original
assert alias is original
assert id(alias) == id(original)
alias.append("green")
print(original)  # ['red', 'blue', 'green']
alias = ["different"]  # Rebinding does not alter original.
print(original, alias)
assert alias is not original


## 4. Equality, identity, and copying

`==` compares values according to the objects' equality behavior. `is` asks whether
two references point to the same object. Use `==` for value comparisons and `is None`
for the absence singleton. `is not` is the negated identity test.

Python implementations may reuse some immutable objects (such as certain integers
or strings). Do not infer value equality from those incidental identity observations.
Two equal lists are an unambiguous demonstration: constructing each separately gives
two distinct objects. A list slice or `.copy()` creates a **shallow copy**: the outer
list is new, but references to objects inside it are copied. Deep copying belongs to
more advanced data-structure work and is not always the desired operation.


In [ ]:
left = [1, 2]
right = [1, 2]
print(left == right, left is right)  # True False
assert 1 == 1.0
assert type(1) is not type(1.0)
missing = None
assert missing is None

nested = [[1], [2]]
shallow = nested.copy()
assert shallow is not nested
assert shallow[0] is nested[0]
shallow[0].append(9)
shallow.append([3])
print("original:", nested, "copy:", shallow)
assert nested == [[1, 9], [2]]


## 5. Namespaces organize bindings

A **namespace** maps names to objects. Modules have namespaces, and a function call
has a local namespace for its parameters and local names. `locals()` and `globals()`
let you inspect these bindings. Reading their results is useful for learning; do not
use edits to a function's `locals()` mapping as a way to assign local variables.

A function's local name can match an outer name without changing that outer binding.
Using namespaces avoids accidental name clashes. Later chapters examine scope and
closures in more detail.


In [ ]:
label = "outside"


def inspect_namespace(value):
    label = "inside"
    local_names = locals()
    return label, local_names["value"], "label" in local_names


print(inspect_namespace(7))  # ('inside', 7, True)
print(label)                 # outside
assert "inspect_namespace" in globals()
assert label == "outside"


## 6. Duck typing: behavior matters

A function can operate on different types when they support the operations it uses.
This is **duck typing**. For `(a + b) * count`, numbers add and multiply, while strings
and lists concatenate and repeat. The same syntax can have different meanings for
different types, so document the intended contract.

Duck typing does not promise that every combination is valid. Adding a list and a
string is unsupported. Catch a specific expected error in a demonstration; avoid a
blanket `except` that would also hide unrelated bugs.


In [ ]:
def combine_and_repeat(a, b, count):
    """Add a and b, then multiply/repeat that result by count."""
    return (a + b) * count


print(combine_and_repeat(1, 2, 3))        # 9
print(combine_and_repeat([1], [2, 3], 2))  # [1, 2, 3, 1, 2, 3]
print(combine_and_repeat("l", "olo", 4))
try:
    combine_and_repeat([1], "two", 2)
except TypeError as error:
    print("Expected incompatible operations:", type(error).__name__)


## 7. String literals and escape sequences

Choose a delimiter that minimizes escaping. `\'` or `\"` includes a matching quote,
`\n` is a newline, `\t` is a tab, and `\\` represents a backslash. A raw string
prefix, as in `r"a\b"`, keeps backslashes literal in common cases; a raw string still
cannot end in a single unescaped backslash. `repr(text)` reveals escapes rather than
rendering them as layout, which helps when inspecting whitespace.


In [ ]:
print("doesn't")
print('"Yes," he said.')
quoted = '"Isn\'t," she said.'
print(quoted)
multiline = "first\nsecond\tcolumn"
print(multiline)
print(repr(multiline))
print(r"folder\notes.txt")


## 8. Search, test, and transform text

String methods return new strings or other results; they do not mutate the original.
Searches are case-sensitive unless you normalize case first. `.find(substring)`
returns the first index or −1 when absent. Use `substring in text` for a direct
membership question: testing `.find()` in an `if` is a trap because index 0 is falsy
and −1 is truthy.

`.replace(old, new)` replaces occurrences. `.startswith(...)` tests a prefix and
`.isalpha()` is true only for a nonempty string of alphabetic characters; spaces and
punctuation make it false. Unicode letters, not only ASCII letters, count.


In [ ]:
greeting = "Hello world! "
print(greeting[4], "world" in greeting, len(greeting))  # o True 13
print(greeting.find("lo"), greeting.find("absent"))    # 3 -1
print(greeting.replace("llo", "y"))                 # Hey world! 
print(greeting.startswith("hell"))                  # False: capital H
print(greeting.isalpha(), "Grüße".isalpha())         # False True
assert greeting == "Hello world! "
assert greeting.find("Hello") == 0
assert "Hello" in greeting


## 9. Case, trimming, splitting, and joining

`.lower()` makes lowercase text; `.casefold()` is useful for caseless Unicode
comparisons (for example, it maps `ß` to `ss`). `.title()` applies title casing,
which is not a complete rule for formatting every person's name or language.
`.strip()` removes surrounding whitespace. `.strip("! ")` removes any of the given
**characters** from both ends; it does not remove one exact prefix or suffix.

`.split()` without a separator splits on runs of whitespace and omits empty pieces.
With an explicit separator, empty fields are preserved. `separator.join(strings)`
joins a sequence of strings; it does not automatically convert numbers to text.


In [ ]:
greeting = "Hello world! "
print(greeting.lower(), greeting.title())
print(repr(greeting.strip()), repr(greeting.strip("! ")))
print("Straße".casefold())  # strasse
print("cabana".strip("abc"))  # nan: removes characters from both ends
print(" ham   cheese\tbacon ".split())
print("03-30-2016".split(sep="-"))
print("a,,b,".split(","))  # ['a', '', 'b', '']
print(", ".join(["Eric", "John", "Michael"]))
try:
    ", ".join(["age", 20])
except TypeError:
    print("Expected TypeError: join needs string elements.")


## 10. Format readable output with f-strings

An f-string evaluates expressions inside braces: `f"{name} scored {points}"`.
Formatting changes the displayed representation, not the stored value. A colon
introduces a format specification: `.2f` means fixed-point with two decimal places;
`06.2f` also requests a minimum width of six padded with zeroes. `<`, `>`, and `^`
request left, right, and centered alignment. The character before an alignment
symbol is the fill character. Double braces produce literal braces.

Widths are minimums, not truncation limits. Rounding for display follows Python's
numeric formatting rules and is not a substitute for a decimal accounting model.


In [ ]:
name, points = "Sam", 7
print(f"{name} scored {points}; squared: {points ** 2}.")
pi_approximation = 3.14159
print(f"{pi_approximation:06.2f}")  # 003.14
print(repr(f"{'hi':10}"))          # 'hi        '
print(f"{'TEST':*^12}")            # ****TEST****
print(f"{'TEST':^12}")             # spaces, not stars
print(f"{{name}} is a placeholder; name is {name}.")
assert pi_approximation == 3.14159


## 11. Read existing `.format` and `%` formatting

`str.format` remains useful when the template is stored separately. It accepts
positional, numbered, and named fields and uses the same format-specification
mini-language as f-strings. Fields can access sequence items.

The `%` operator also performs older string interpolation with placeholders such as
`%s` and `%d`. Recognize it when reading existing code. F-strings are a clear default
for local expressions in modern Python; no formatting method is universally fastest.
Concatenation works too, but non-string values require conversion.


In [ ]:
print("{} {}".format("monty", "python"))
print("{0} can be {1} {0}s".format("strings", "formatted"))
print("{name} loves {food}".format(name="Sam", food="plums"))
print("{} squared is {}".format(5, 5 ** 2))
print("{:06.2f}".format(3.14159))
captains = ["Kirk", "Picard"]
print("{caps[0]} > {caps[1]}".format(caps=captains))
print("%s, %s, %s. (Act %d)" % ("Words", "words", "words", 2))
age = 20
assert "I am " + str(age) + " years old." == f"I am {age} years old."


## 12. Files, paths, modes, and encodings

A file object mediates reading and writing. Relative paths are interpreted from the
current working directory, which need not be the notebook's directory; absolute paths
specify a complete location. `pathlib.Path` makes path construction readable.

| Mode | Purpose | Important behavior |
|---|---|---|
| `r` | read text (default) | missing file raises `FileNotFoundError` |
| `w` | write text | creates a file or truncates an existing file |
| `a` | append text | preserves existing content and writes at the end |
| `x` | create text exclusively | raises `FileExistsError` if it exists |
| `rb`, `wb` | read/write bytes | no text decoding or encoding |

Specify `encoding="utf-8"` for text. Use `with ... as ...` to close the file reliably
when the block exits. Every example below uses a fresh temporary directory, so it
cannot overwrite a personal file and does not rely on a previous cell's file.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix="python-fundamentals-") as directory:
    path = Path(directory) / "notes.txt"
    with path.open("w", encoding="utf-8") as stream:
        written = stream.write("Grüße\n")
        stream.writelines(["Python\n", "files\n"])
        stream.flush()  # Push Python buffers onward; not a durable-disk guarantee.
    assert stream.closed
    with path.open("r", encoding="utf-8") as stream:
        content = stream.read()
    print(repr(content), "characters in first write:", written)
    with path.open("a", encoding="utf-8") as stream:
        stream.write("appended\n")
    print(path.read_text(encoding="utf-8"))
# The context has removed the temporary directory and its contents.
assert not path.exists()


## 13. Reading strategies and cursor position

`.read()` reads the remaining content, `.read(size)` reads up to that many characters
in text mode, `.readline()` reads one line, and `.readlines()` makes a list of the
remaining lines. Iterating directly over the file processes lines one at a time.
Each operation advances the file's position, so repeated reads need not return the
same data. At end-of-file, text `.read()` and `.readline()` return `""`.

Newline characters remain in returned lines. `.writelines(...)` does **not** add them
for you. `print(line, end="")` avoids adding a second newline when printing an
already-terminated line. For large files, line iteration avoids loading everything
into memory at once.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "lines.txt"
    path.write_text("alpha\nbeta\ngamma\n", encoding="utf-8")
    with path.open(encoding="utf-8") as stream:
        print(repr(stream.read(2)))     # 'al'
        print(repr(stream.readline()))  # 'pha\n'
        print(stream.readlines())       # ['beta\n', 'gamma\n']
        assert stream.read() == ""
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            print(line, end="")
    with path.open("rb") as stream:
        raw_bytes = stream.read()
    assert isinstance(raw_bytes, bytes)
    print(repr(raw_bytes))


## 14. Cleanup also happens when an error occurs

Manual `open` followed later by `close` can leave a file unclosed if an exception
interrupts the code before `close`. A context manager calls its exit behavior even
when the body raises. File context managers close the file; they do not silently
ignore errors. Catching an expected error **outside** the `with` makes this visible.

A `with` block does not create a new variable scope. A name such as `content` remains
bound after the block if it was successfully assigned, although the file is closed.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "safe.txt"
    path.write_text("available", encoding="utf-8")
    try:
        with path.open(encoding="utf-8") as stream:
            content = stream.read()
            raise ZeroDivisionError("A deliberate example failure")
    except ZeroDivisionError:
        print("Expected failure; the file has still been closed.")
    assert stream.closed
    assert content == "available"
    assert "content" in locals()
    try:
        (Path(directory) / "missing.txt").read_text(encoding="utf-8")
    except FileNotFoundError:
        print("Expected FileNotFoundError for a path we did not create.")


## 15. Imports give access to module namespaces

`import math` binds the module name, so `math.sqrt(...)` clearly shows where the
function comes from. `from math import ceil` binds a selected name directly.
`import math as mathematics` assigns a local alias. Choose a style that stays clear
and follows the project convention; importing the whole module is not a universal
requirement. Avoid `from module import *`, which makes name origins hard to track.

`math.sqrt(16)` returns the float `4.0`. For ordinary floating inputs, `math.ceil`
and `math.floor` return integers, correcting the float outputs shown in the slides.
Importing a module normally executes its top-level code once per interpreter session;
Python then caches that module in `sys.modules`.


In [ ]:
import math
from math import ceil, floor
import math as mathematics

print(math.sqrt(16))  # 4.0
print(ceil(3.7), floor(3.7))  # 4 3
assert type(ceil(3.7)) is int
assert mathematics is math
print(math.__name__)


## 16. A Python file can be a script and an importable module

A `.py` file on the import search path can be a module. Running a file directly sets
its `__name__` to `"__main__"`; importing it sets `__name__` to its module name.
A **main guard** keeps demonstrations from running merely because somebody imports
a useful function:

```python
def square(value):
    return value * value

if __name__ == "__main__":
    print(square(4))
```

The next cell creates that module temporarily and uses two fresh Python processes:
one executes it as a script, and one imports it. The import prints nothing by itself;
the explicit call in the importing program prints 25. `sys.executable` chooses the
same interpreter as this notebook, and a timeout prevents an accidental indefinite
wait. The temporary directory supplies the import search location for that process.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import subprocess
import sys

module_source = """def square(value):
    return value * value

if __name__ == "__main__":
    print(square(4))
"""
with TemporaryDirectory() as directory:
    module_path = Path(directory) / "lesson_math.py"
    module_path.write_text(module_source, encoding="utf-8")
    scripted = subprocess.run(
        [sys.executable, str(module_path)],
        capture_output=True, text=True, check=True, timeout=10,
    )
    imported = subprocess.run(
        [sys.executable, "-c", "import lesson_math; print(lesson_math.square(5))"],
        cwd=directory, capture_output=True, text=True, check=True, timeout=10,
    )
    print("Script output:", scripted.stdout.strip())
    print("Import plus explicit call:", imported.stdout.strip())
    assert scripted.stdout == "16\n"
    assert imported.stdout == "25\n"


## 17. Virtual environments and package installation

A virtual environment isolates a project's interpreter environment and installed
packages from other projects. It does not isolate the filesystem, remove the need
for dependency versions, or automatically change a notebook's active kernel.
`sys.executable` shows which interpreter is actually running; `sys.prefix` and
`sys.base_prefix` often help identify a standard `venv` environment.

Use the repository setup notebook and README for the complete supported workflow.
The primary repository workflow is `uv sync --locked`. As a standalone alternative,
standard-library tools can create an environment with Jupyter:

```sh
python3 -m venv .venv
# macOS/Linux: source .venv/bin/activate
# Windows PowerShell: .venv\Scripts\Activate.ps1
python -m pip install jupyterlab ipykernel
```

`python -m pip` associates pip with the selected interpreter. `uv` can also create
and manage environments using the documented project workflow. Environment activation
adjusts command lookup in that terminal; you can instead call an environment's Python
by its explicit path. Select the corresponding Jupyter kernel separately.

The original Python 3.4, Python 2, `virtualenvwrapper`, and `easy_install` setup reflects
its historical context. This repository pins Python 3.13.15. A per-project
`venv` or `uv` environment is sufficient; no old wrapper tool is required. Conda is
also a valid ecosystem, particularly for mixed-language scientific dependencies;
blanket claims that it is less supported are not a useful selection rule. Install
packages into the intended environment rather than modifying a system Python.


In [ ]:
import sys

print("Active Python:", sys.executable)
print("Version:", sys.version.split()[0])
print("Prefix:", sys.prefix)
print("Base prefix:", sys.base_prefix)
print("Standard venv detected:", sys.prefix != sys.base_prefix)
# This cell reports the current kernel; it does not install or change anything.


## Practice

Each task is independent. Build a result you can inspect, then check the stated
edge cases. The separate solutions notebook includes all imports and setup it uses.

### Exercise 1 · Predict shared references

Create `original = [["red"], ["blue"]]`, `alias = original`, and `shallow = original.copy()`.
Append `"green"` to `shallow[0]` and append `["black"]` to `shallow`.
Write assertions showing which outer lists are identical, which first inner lists are
identical, and why `original` has two items while `shallow` has three. Finally rebind
`alias` to `[]` and verify that `original` still contains its data.


In [ ]:
original = [["red"], ["blue"]]
# TODO: create the two references, make changes, and check the observations.


### Exercise 2 · One function, several types

Define `combine_and_repeat(a, b, count)` to return `(a + b) * count`. Check integers,
strings, and lists: `(2, 3, 4)` gives 20; `("py", "thon", 2)` gives `"pythonpython"`;
`([1], [2], 0)` gives `[]`. Show that incompatible inputs `([1], "2", 1)` raise
`TypeError`, catching that specific error so execution continues. Explain why zero
has a different-looking result for numbers and sequences.


In [ ]:
def combine_and_repeat(a, b, count):
    # TODO: implement the operation and add successful/error checks.
    pass


### Exercise 3 · Normalize whitespace and case

Define `normalize_words(text)` to split on whitespace, case-fold each word, and
join the words with one space. For `"  PYTHON\t Straße \n"`, return `"python strasse"`.
For empty or whitespace-only text, return `""`. Keep punctuation: `"Hello, WORLD!"`
becomes `"hello, world!"`. Do not assume `.strip()` removes internal whitespace.


In [ ]:
def normalize_words(text):
    # TODO: split, case-fold the pieces, and join them.
    pass


### Exercise 4 · An aligned report row

Define `format_report_row(name, count, mean)` returning an f-string with name
left-aligned in width 10, count right-aligned in width 4, and mean in width 7 with
2 decimals. Separate fields using `" | "`. Example:
`format_report_row("A", 3, 2.5)` must return `"A          |    3 |    2.50"`.
Check a zero count, a negative mean, and a name longer than 10 characters. Do not
truncate a long name. Reproduce the normal example with `.format()` as a reading exercise.


In [ ]:
def format_report_row(name, count, mean):
    # TODO: specify alignment, width, and numeric precision.
    pass


### Exercise 5 · A safe UTF-8 note round-trip

Define `write_notes(path, notes)` to write each string in `notes` as one newline-ended
line using UTF-8, and `read_notes(path)` to return those lines with **only the terminal
newline** removed. Assume individual notes contain no newline characters; preserve
other spaces and allow an empty note. Use `with` for opened files.

Test in a `TemporaryDirectory` with `["Grüße", "  keep spaces  ", ""]`, then with an
empty list. Verify that rewriting with mode `w` replaces the old content. Check a
missing file by catching `FileNotFoundError`. The temporary path must not exist after
its context ends.


In [ ]:
def write_notes(path, notes):
    # TODO: use pathlib.Path, open(..., encoding="utf-8"), and write newlines.
    pass


def read_notes(path):
    # TODO: return line contents while preserving meaningful whitespace.
    pass


### Exercise 6 · Make import behavior predictable

Create a temporary module `temperature_tools.py` with a function
`celsius_to_fahrenheit(celsius)` and a main guard that prints the conversion of 0.
Use `subprocess.run` and `sys.executable` to run the file and then import it from a
fresh process whose working directory is the temporary directory. The script should
print `32.0`; an import on its own should print nothing; an explicit imported call
with −40 should print `-40.0`. Use `check=True`, captured output, and a timeout.
Keep all files inside the temporary context.


In [ ]:
# TODO: import pathlib, tempfile, subprocess, and sys as needed.
# TODO: write the module source in a temporary directory.
# TODO: check script output, quiet import, and explicit imported call.


## Check your understanding

Explain when two names share an object, why a shallow copy can share nested data,
when to use `is None`, and why a successful duck-typed call says nothing about every
possible argument combination. Can you predict the difference between `strip` and
`split`, or between a file's first and second `read`? Explain what a main guard protects
and how you would identify the interpreter used by a notebook.

**Continue:** the next chapter explores Python collections in more depth.
